In [ ]:
import shutil, os
# Copy the script to your working directory if not already there
# (assumes train_single_task.py is in the same folder as this notebook)
assert os.path.exists("train_single_task.py"), \
    "train_single_task.py not found — make sure it's in the same directory as the notebook"
print("train_single_task.py found")

In [ ]:
# Standard Library
import os
import sys
import json
import math
import random
from pathlib import Path
from typing import List, Optional, Tuple, Union
from collections import defaultdict

# Data Science & Utilities
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# DeepChem
import deepchem
from deepchem.molnet import *

# RDKit
from rdkit import Chem

# Torch
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn import CrossEntropyLoss, MSELoss, BCEWithLogitsLoss
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast

# Accelerate
from accelerate import Accelerator

# Transformers
from transformers import (
    AutoModel, AutoModelForMaskedLM, AutoModelForSequenceClassification,
    AutoTokenizer, RobertaModel, RobertaTokenizer, RobertaTokenizerFast,
    RobertaConfig, RobertaPreTrainedModel, DebertaV2Model, DebertaV2Config,
    DebertaTokenizerFast, LongformerModel, LongformerConfig,
    LongformerForMaskedLM, LongformerTokenizerFast,
    AutoConfig, Trainer, TrainingArguments, pipeline,
    default_data_collator
)
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers.trainer_pt_utils import get_parameter_names

# Tokenizers
from tokenizers import ByteLevelBPETokenizer
from tokenizers.implementations import ByteLevelBPETokenizer
from tokenizers.processors import BertProcessing

# ESM
import esm

# Datasets
from datasets import Dataset, DatasetDict, load_from_disk

# Metrics
import sklearn
from sklearn.metrics import (
    root_mean_squared_error, mean_squared_error, roc_auc_score
)

# Bitsandbytes
import bitsandbytes as bnb

# Misc
import ipynbname


In [ ]:
def get_notebook_name():
    try:
        return ipynbname.name()
    except Exception:
        return "unknown_notebook"

# Get the notebook's name
notebook_name = get_notebook_name().split(".")[0]  # Remove file extension if any
os.makedirs("result_graphs", exist_ok=True)

In [ ]:
model_pth = "../Pre-Training/PubchemModelDeberta"
model_pth2 = "../Pre-Training/ProteinModelDeb"

In [ ]:
updated_datasets = {"train": {}, "test": {}}

In [ ]:
CONVERTED_DIR   = "data/newTrainVal"           # output of convert_uniprot_to_sequence.py
SAVE_DIR        = "Data3/inputdataMMAT"        # where to save processed datasets
TRAIN_SAVE_PATH = os.path.join(SAVE_DIR, "train_datasets")
TEST_SAVE_PATH  = os.path.join(SAVE_DIR, "test_datasets")
TASKS_PATH      = os.path.join(SAVE_DIR, "tasks.json")
mmat_exists = (
    os.path.exists(TRAIN_SAVE_PATH) and
    os.path.exists(TEST_SAVE_PATH)  and
    os.path.exists(TASKS_PATH)
)

if mmat_exists:
    print("MMAT datasets already saved — loading from disk.")
    mmat_train = dict(load_from_disk(TRAIN_SAVE_PATH))
    mmat_test  = dict(load_from_disk(TEST_SAVE_PATH))
    with open(TASKS_PATH) as f:
        mmat_tasks = json.load(f)

In [ ]:
# Unified task and dataset details
Datasets_Available = {
}
for task_name, ds in mmat_train.items():
    updated_datasets["train"][task_name] = ds

for task_name, ds in mmat_test.items():
    updated_datasets["test"][task_name] = ds
    
print("\nupdated_datasets['train'] keys:", list(updated_datasets["train"].keys()))
print("\nSpot-check features per dataset type:")
for name, ds in updated_datasets["train"].items():
    print(f"  {name:30s}  rows={len(ds):6d}  cols={ds.column_names}")


for task_name in mmat_tasks:
    Datasets_Available[task_name] = {
        "problem_type": "regression",
        "num_labels": 1,
        "used_encoders": "multiencoder"
    }

num_labels_list   = [v["num_labels"]    for v in Datasets_Available.values()]
problem_type_list = [v["problem_type"]  for v in Datasets_Available.values()]

print("\nDatasets_Available keys:", list(Datasets_Available.keys()))
print("num_labels_list:", num_labels_list)
num_labels_list = [value["num_labels"] for value in Datasets_Available.values()]
problem_type_list = [value["problem_type"] for value in Datasets_Available.values()]

tokenizer = DebertaTokenizerFast.from_pretrained(model_pth)
tokenizer2 = DebertaTokenizerFast.from_pretrained(model_pth2)
tokenizer

In [ ]:
import subprocess, time, sys
from datetime import timedelta

def monitor(processes: dict, poll_interval: int = 30, wave_label: str = ""):
    pending   = dict(processes)
    results   = {}
    wave_start = time.time()

    print(f"\n{' '*60}")
    print(f"Monitoring wave{(' ' + wave_label) if wave_label else ''}: "
          f"{list(pending.keys())}")
    print(f"{' '*60}")

    while pending:
        time.sleep(poll_interval)
        finished = []
        for task_name, proc in pending.items():
            ret = proc.poll()
            if ret is not None:
                elapsed = timedelta(seconds=int(time.time() - wave_start))
                status  = "Processed" if ret == 0 else f" exit={ret}"
                print(f"  {status}  {task_name:<30s}  elapsed={elapsed}")
                if ret != 0:
                    print(f"     ↳ check logs: task_logs/{task_name}.log")
                results[task_name] = ret
                finished.append(task_name)

        for t in finished:
            del pending[t]

        if pending:
            elapsed = timedelta(seconds=int(time.time() - wave_start))
            print(f"  ⏳ still running ({elapsed}): {list(pending.keys())}")

    wave_mins = (time.time() - wave_start) / 60
    print(f"\n  Wave done in {wave_mins:.1f} min")
    return results


def launch_task(task_name: str, gpu_id: int, save_dir: str,
                notebook_name: str, log_dir: str) -> subprocess.Popen:
    """Spawn a single training subprocess pinned to gpu_id."""
    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, f"{task_name}.log")
    log_file = open(log_path, "w")
    proc = subprocess.Popen(
        [sys.executable, "train_single_task.py",
         task_name, str(gpu_id), save_dir, notebook_name],
        stdout=log_file,
        stderr=log_file,
    )
    print(f"  {task_name:<30s} → GPU {gpu_id}  pid={proc.pid}  log={log_path}")
    return proc

In [ ]:
NUM_GPUS     = 1
SAVE_DIR_MLT = "../Models/Stratified_CheMLT-F"
LOG_DIR      = "task_logs_tune"
os.makedirs("task_results_tune", exist_ok=True)

task_names  = list(updated_datasets["train"].keys())
all_results = {}          # task_name → return_code
wall_start  = time.time()

# Split tasks into waves of NUM_GPUS
waves = [task_names[i : i + NUM_GPUS] for i in range(0, len(task_names), NUM_GPUS)]

print(f"Tasks : {len(task_names)}")
print(f"GPUs  : {NUM_GPUS}")
print(f"Waves : {len(waves)}")
for wi, w in enumerate(waves):
    print(f"  Wave {wi+1}: {w}")

for wave_idx, wave_tasks in enumerate(waves):
    print(f"\n{'═'*60}")
    print(f"  WAVE {wave_idx + 1} / {len(waves)}")
    print(f"{'═'*60}")
    wave_procs = {}

    for gpu_id, task_name in enumerate(wave_tasks):
        proc = launch_task(task_name, gpu_id, SAVE_DIR_MLT, notebook_name, LOG_DIR)
        wave_procs[task_name] = proc

    wave_results = monitor(wave_procs, poll_interval=30, wave_label=str(wave_idx + 1))
    all_results.update(wave_results)

total_wall = time.time() - wall_start
print(f"\n{'═'*60}")
print(f"  ALL WAVES DONE  —  total wall time: {total_wall/60:.1f} min")
print(f"{'═'*60}")

failed = [t for t, rc in all_results.items() if rc != 0]
if failed:
    print(f"\n  Failed tasks: {failed}")
    print("   Check their logs in task_logs_tune/<task>.log")
else:
    print("\n All tasks finished successfully")


In [ ]:
import torch, copy
from collections import defaultdict

train_metrics_arr = {}
eval_metrics_arr  = {}

for task_name in task_names:
    result_path = f"task_results_tune/{task_name}_metrics.pt"
    if os.path.exists(result_path):
        result = torch.load(result_path, weights_only=False)
        train_metrics_arr[task_name] = result["train"]
        eval_metrics_arr[task_name]  = result["eval"]
        print(f"Complete {task_name}")
    else:
        print(f"Err {task_name}  — results file missing, check task_logs/{task_name}.log")

print("\ntrain_metrics_arr keys:", list(train_metrics_arr.keys()))
print("eval_metrics_arr  keys:", list(eval_metrics_arr.keys()))
